# Lab 2 | Hubble Space Explorer
**Find bright celestial-source candidates in a real telescope image.**

**Dataset:** the Hubble eXtreme Deep Field image, originally from HubbleSite/NASA and distributed in `scikit-image`.

**Goal:** train K-means to group pixels by color and brightness, then locate bright regions. A detection is a **candidate**, not confirmation that an object is a galaxy or star.

**Instructions:** Runtime → Run all. Modify `NUMBER_OF_GROUPS` and `MIN_REGION_AREA`, then compare.

### 1 | Explore the Hubble image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, transform, measure
from sklearn.cluster import MiniBatchKMeans
from matplotlib.patches import Circle

# Real Hubble eXtreme Deep Field image; no separate download needed.
original = data.hubble_deep_field()
image = transform.resize(original, (400, 458), anti_aliasing=True)
print("Hubble image shape:", original.shape)
plt.figure(figsize=(9, 6))
plt.imshow(original); plt.title("Hubble eXtreme Deep Field")
plt.axis("off"); plt.show()

### 2 | Train a model to identify groups of similar pixels

In [ ]:
# GROUP CHALLENGE: try 3, 4, or 5 color groups.
NUMBER_OF_GROUPS = 4

# Use some of the image pixels as training examples to keep it fast.
pixels = image.reshape(-1, 3)
random = np.random.default_rng(7)
training_pixels = pixels[random.choice(len(pixels), 25000, replace=False)]
model = MiniBatchKMeans(n_clusters=NUMBER_OF_GROUPS, random_state=7,
                        n_init=3, batch_size=2048)
model.fit(training_pixels)
labels = model.predict(pixels).reshape(image.shape[:2])

# Compute perceived brightness of each learned group.
brightness = model.cluster_centers_ @ np.array([0.2126, 0.7152, 0.0722])
brightest_group = int(np.argmax(brightness))
bright_mask = labels == brightest_group
print("Learned brightness values:", np.round(brightness, 3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image); axes[0].set_title("Original Hubble image")
axes[1].imshow(bright_mask, cmap="gray")
axes[1].set_title("Brightest AI color group")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

### 3 | Locate and label the brightest source candidates

In [ ]:
# GROUP CHALLENGE: increase 8 to 15 or 25 to ignore smaller regions.
MIN_REGION_AREA = 8

# Connected pixels form a candidate region.
connected = measure.label(bright_mask)
regions = [r for r in measure.regionprops(connected) if r.area >= MIN_REGION_AREA]
regions = sorted(regions, key=lambda r: r.area, reverse=True)

# Display the 20 largest bright regions with circles.
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
for region in regions[:20]:
    cy, cx = region.centroid
    radius = max(6, np.sqrt(region.area / np.pi) + 3)
    ax.add_patch(Circle((cx, cy), radius, fill=False, color="yellow", linewidth=1.2))
ax.set_title(f"Bright-source candidates: {len(regions)} regions | top 20 marked")
ax.axis("off"); plt.show()
print(f"Candidate bright regions: {len(regions)}")
print("Remember: a bright region is not automatically a galaxy or star.")

### Team mission
1. Compare `NUMBER_OF_GROUPS = 3`, `4`, and `5`. How does the number of candidates change?
2. Increase `MIN_REGION_AREA` from `8` to `20`. Which objects disappear?
3. Choose a region that looks like a false alarm and explain why.
4. Prepare a **30-second demonstration** with the annotated telescope image.

**Debrief:** Why does bright-object detection require scientific verification?